# Surrogacy Case Audit
Shows every file counted in the monthly statistics, one row per case.
Run all cells top-to-bottom, then use the filter cells at the bottom to inspect.

In [53]:


import os, re
from datetime import datetime
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 300)

BASE_DIR = r"C:\Users\zhous\OneDrive - Tsong Law Group\Ralph Tsong's files - Active Cases\Surrogacy Cases"

In [54]:
# ── Core functions (kept in sync with surrogacy_monthly_final_v2.py) ──────────

# ── Agency canonical names & aliases ──────────────────────────────────────────
# Keys = lowercase variant seen in folder names; values = canonical display name.
# Add more rows here as needed — no other code needs to change.
AGENCY_CANONICAL = {
    # Giving Tree Surrogacy
    'gts':                                         'Giving Tree Surrogacy',
    'giving tree':                                 'Giving Tree Surrogacy',
    'giving tree surrogacy (review)':              'Giving Tree Surrogacy',

    # Royal Surrogacy & Egg Donation
    'royal surrogacy':                             'Royal Surrogacy & Egg Donation',
    'royal':                                       'Royal Surrogacy & Egg Donation',

    # Coast to Coast Surrogacy
    'coast to coast':                              'Coast to Coast Surrogacy',

    # Patriot Conceptions  ← note: do not mention agency name to client
    'patriot conceptions (do not mention agency)': 'Patriot Conceptions',
    'patriot':                                     'Patriot Conceptions',

    # LAS / Los Angeles Surrogacy
    'los angeles surrogacy':                       'LAS',
    'la surrogacy':                                'LAS',

    # All Families Surrogacy
    'all families':                                'All Families Surrogacy',
    'all family surrogacy':                        'All Families Surrogacy',
    'afs':                                         'All Families Surrogacy',

    # Blossom California Fertility
    'blossom california':                          'Blossom California Fertility',
    'blossom':                                     'Blossom California Fertility',

    # California Surrogacy Partners
    'csp':                                         'California Surrogacy Partners',

    # Simple Surrogacy
    'simple':                                      'Simple Surrogacy',

    # Oneworld Generations
    'oneworld generations surrogacy':              'Oneworld Generations',
    'oneworld':                                    'Oneworld Generations',

    # Modern Baby Family
    'modern baby family corp':                     'Modern Baby Family',
    'modern baby family':                          'Modern Baby Family',

    # Lily Baby Surrogacy
    'lily baby surrogacy':                         'Lily Baby',

    # SurrogateFirst
    'surrogatefirst':                              'SurrogateFirst',

    # Surrogacy4All
    'surrogacy4all':                               'Surrogacy4All',

    # Global Fertility
    'global fertility':                            'Global Fertility',

    # Dream Surrogacy
    'dream surrogacy':                             'Dream Surrogacy',

    # Blissful Bloom
    'blissful bloom':                              'Blissful Bloom',

    # ACRC
    'acrc':                                        'ACRC',

    # EDSI
    'edsi':                                        'EDSI',

    # Indy (independent / no agency)
    'indy':                                        'Indy',

    # Newgen Families
    'newgen families':                             'Newgen Families',
    'new gen families':                            'Newgen Families',
    'new gen':                                     'Newgen Families',
    'newgen':                                      'Newgen Families',
    'new generation families':                     'Newgen Families',
    'new generation':                              'Newgen Families',
}

# Special notes attached to canonical agency names (shown in exports)
AGENCY_NOTES = {
    'Patriot Conceptions': 'Do not mention agency name to client',
}


def canonicalize_agency(name):
    if not name:
        return name
    key = name.strip().lower()
    if key in AGENCY_CANONICAL:
        return AGENCY_CANONICAL[key]
    for variant, canonical in AGENCY_CANONICAL.items():
        if key == variant or key.startswith(variant + ' ') or variant.startswith(key + ' '):
            return canonical
    return name


def normalize_agency_name(agency_name):
    if "(do not use)" in agency_name.lower():
        return None
    normalized = re.sub(r'^review\s+', '', agency_name, flags=re.IGNORECASE)
    normalized = re.sub(r'^\s*\([A-Z]{2}\)\s*', '', normalized)
    normalized = re.sub(r'^-\s*', '', normalized)
    normalized = re.sub(r'\s*\([A-Z]{2}\)\s*$', '', normalized)
    normalized = re.sub(r',?\s*(Inc\.?|LLC|PLLC|Corp\.?|Ltd\.?)\s*$', '', normalized, flags=re.IGNORECASE)
    words = normalized.split()
    if len(words) >= 3 and words[-1].lower() in ['surrogacy', 'donation']:
        normalized = ' '.join(words[:-1])
    elif len(words) >= 2 and words[-1].lower() in ['nmc', 'break', 'review']:
        normalized = ' '.join(words[:-1])
    normalized = re.sub(r'\s+(Consulting|will change|for|do not)\s+.*$', '', normalized, flags=re.IGNORECASE)
    normalized = ' '.join(normalized.split()).strip()
    if not normalized:
        return None
    # reject bare 2-letter state codes (e.g. "WA", "CA") leaking through as agency names
    if re.match(r'^[A-Z]{2}$', normalized):
        return None
    return canonicalize_agency(normalized)


def parse_folder_name(folder_name):
    if not (folder_name.startswith('24-') or folder_name.startswith('25-') or folder_name.startswith('26-')):
        return None
    is_review = "review" in folder_name.lower()
    clean_name = re.sub(r'\s*-\s*(TERMINATE4D|TERMINATED)\s*$', '', folder_name, flags=re.IGNORECASE).strip()
    name_part = re.sub(r'^2[456]-\S+\s+', '', clean_name, flags=re.IGNORECASE).strip()
    agency = None
    parts = [p.strip() for p in clean_name.split(' - ')]
    if len(parts) >= 2:
        agency = parts[-1]
    if not agency:
        norm = re.sub(r'\s*[–—]\s*', ' - ', name_part)
        norm = re.sub(r'(?<!\s)-\s+', ' - ', norm)
        norm = re.sub(r'\s+-([^\s-])', r' - \1', norm)
        parts = [p.strip() for p in norm.split(' - ')]
        if len(parts) >= 2:
            agency = parts[-1]
    if not agency and is_review:
        m = re.search(r'\breview\s+([^\s].+?)\s*$', name_part, re.IGNORECASE)
        if m:
            agency = m.group(1).strip()
    if not agency and '&' in name_part:
        after_amp = name_part.split('&')[-1].strip()
        after_amp = re.sub(r'\s*\([A-Z]{2,3}(?:-[A-Z]{2,3})?\)\s*', ' ', after_amp).strip()
        after_amp = re.sub(r'\s*\b(PBO|nmc|review|clean\s+up)\s*$', '', after_amp, flags=re.IGNORECASE).strip()
        after_amp = re.sub(r'^(\w[\w\-]+(?:\s+and\s+\w[\w\-]+)?)\s+', '', after_amp).strip()
        after_amp = re.sub(r'\s*\b(PBO|nmc|review)\s*$', '', after_amp, flags=re.IGNORECASE).strip()
        if after_amp:
            agency = after_amp
    if agency:
        agency = agency.strip()
        if agency.upper() not in ['XX', 'XXX', 'XXXX']:
            return (agency, is_review)
    return None


def find_agreement_file(folder_path):
    try:
        for entry in os.scandir(folder_path):
            if not entry.is_file():
                continue
            n = entry.name.lower()
            if any(n.endswith(ext) for ext in ['.pdf', '.jpg', '.jpeg', '.png', '.gif', '.tiff', '.bmp']):
                if 'agreement' in n and 'legal' in n and 'representation' in n and 'signed' in n:
                    return entry
    except:
        pass
    return None


def find_clearance_letter(folder_path):
    try:
        for entry in os.scandir(folder_path):
            if not entry.is_file():
                continue
            n = entry.name.lower()
            if 'clearance' in n and any(n.endswith(e) for e in ['.pdf', '.jpg', '.jpeg', '.png', '.gif', '.tiff', '.bmp']):
                return entry
    except:
        pass
    return None

print('Functions loaded OK')
print(f'Agency aliases: {len(AGENCY_CANONICAL)}  |  Agency notes: {len(AGENCY_NOTES)}')

Functions loaded OK
Agency aliases: 36  |  Agency notes: 1


In [55]:
# ── Scan and collect one record per counted case ───────────────────────────────

_SKIP_PATTERN = re.compile(
    r'\b(TERMINATE4D|TERMINATED|NOT\s+DRAFTED\s+YET|DECIDE\s+NOT\s+TO\s+REPRESENT|DECLINED)\b',
    re.IGNORECASE
)

records = []

for folder_name in sorted(os.listdir(BASE_DIR)):
    if folder_name.startswith('.') or folder_name.startswith('~'):
        continue
    folder_path = os.path.join(BASE_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue

    # skip terminated / irrelevant cases entirely
    if _SKIP_PATTERN.search(folder_name):
        continue

    parsed = parse_folder_name(folder_name)
    if not parsed:
        continue

    raw_agency, is_review = parsed
    agency = normalize_agency_name(raw_agency)
    if not agency:
        continue

    entry = find_agreement_file(folder_path)
    source = 'signed agreement'
    if not entry:
        entry = find_clearance_letter(folder_path)
        source = 'legal clearance letter'
    if not entry:
        continue

    try:
        mtime = entry.stat().st_mtime
        agreement_date = datetime.fromtimestamp(mtime)
    except:
        continue

    records.append({
        'month':          agreement_date.strftime('%Y-%m'),
        'date':           agreement_date.strftime('%Y-%m-%d'),
        'agency':         agency,
        'status':         'review' if is_review else 'drafting',
        'case_folder':    folder_name,
        'agreement_file': entry.name,
        'source':         source,
        'path_len':       len(entry.path),
    })

df = pd.DataFrame(records).sort_values(['month', 'agency', 'date']).reset_index(drop=True)
print(f'Total cases counted: {len(df)}')
print(f'  via signed agreement:       {(df["source"]=="signed agreement").sum()}')
print(f'  via legal clearance letter: {(df["source"]=="legal clearance letter").sum()}')
print(f'Months covered:      {df["month"].nunique()}  ({df["month"].min()} → {df["month"].max()})')
print(f'Agencies found:      {df["agency"].nunique()}')
print(f'Drafting:            {(df["status"]=="drafting").sum()}')
print(f'Review:              {(df["status"]=="review").sum()}')

Total cases counted: 1090
  via signed agreement:       1004
  via legal clearance letter: 86
Months covered:      30  (2024-01 → 2026-06)
Agencies found:      183
Drafting:            769
Review:              321


## Full listing — every counted case

In [56]:
df[['month', 'date', 'agency', 'status', 'case_folder', 'agreement_file']]

,month,date,agency,status,case_folder,agreement_file
0,2024-01,2024-01-20,ACRC,drafting,24-S-09 Wang and Jiang & Haug (MN) - ACRC,(Signed) Agreement for Legal Representation - MN - Wang and Jiang & Haug - ACRC.pdf
1,2024-01,2024-01-11,All Families Surrogacy,review,24-S-05 Santelli and Hagood & Severn - Review (WA) All Families Surrogacy,Agreement for Legal Representation - Santelli and Hagood & Severn review (WA) AFS - signed.pdf
2,2024-01,2024-01-22,California Surrogacy Partners,review,24-S-08 Latner and Feldstein & Freytag - Review CSP,Agreement for Legal Representation - Latner and Feldstein & Freytag - Review - CSP - IM Signed.pdf
3,2024-01,2024-01-08,Giving Tree Surrogacy,review,24-S-03 Ding & Martinez - Giving Tree Surrogacy (Review),Agreement for Legal Representation- Martinez GTS review - signed.pdf
4,2024-01,2024-01-18,Giving Tree Surrogacy,review,24-S-07 Watters and Mastrodonato & Carli - Review - GTS,Agreement for Legal Representation - Watters and Mastrodonato & Carli - Review - GTS - signed.pdf
...,...,...,...,...,...,...
1085,2026-06,2026-06-10,One Village,drafting,26-S-283 Hung & Ward (KY) - One Village Surrogacy,请签署Agreement for Legal Representation - Hung & Ward (KY) - One Village Surrogacy - signed.pdf
1086,2026-06,2026-06-05,Oneworld Generations,drafting,26-S-279 Zhang and Wu & Brown - Oneworld Generations Surrogacy,请签署 - Agreement for Legal Representation - Zhang and Wu & Brown - Oneworld Generations Surrogacy - signed.pdf
1087,2026-06,2026-06-02,Surrogacy4All,review,26-S-273 Dean and Mangla & Good (OK) - Surrogacy4All review,Agreement for Legal Representation - Dean and Mangla & Good (OK) - Surrogacy4All review - signed.pdf
1088,2026-06,2026-06-05,SurrogateFirst,drafting,26-S-265 Xiao and Mi & Simbe (UT) - SurrogateFirst NMC,Agreement for Legal Representation - Xiao and Mi & Simbe (UT) - SurrogateFirst - signed.pdf


## Monthly summary (pivot)

In [57]:
pivot_month = (
    df.groupby(['month', 'status'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=['drafting', 'review'], fill_value=0)
)
pivot_month['total'] = pivot_month.sum(axis=1)
pivot_month.loc['TOTAL'] = pivot_month.sum()
pivot_month

status,drafting,review,total
month,,,
2024-01,6,6,12
2024-02,10,4,14
2024-03,15,7,22
2024-04,9,10,19
2024-05,13,3,16
2024-06,11,3,14
2024-07,18,6,24
2024-08,20,5,25
2024-09,19,8,27


## Agency summary (all months combined)

In [58]:
pivot_agency = (
    df.groupby(['agency', 'status'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=['drafting', 'review'], fill_value=0)
)
pivot_agency['total'] = pivot_agency.sum(axis=1)
pivot_agency.sort_values('total', ascending=False)

status,drafting,review,total
agency,,,
LAS,178,24,202
Blossom California Fertility,24,65,89
SurrogateFirst,79,3,82
Giving Tree Surrogacy,38,21,59
Indy,13,16,29
Twinkle Surrogacy,28,0,28
Flying Stork,23,0,23
C&T Fertility Consultant,2,20,22
Lily Baby,21,0,21


## Agency × Month breakdown

In [59]:
pivot_full = (
    df.groupby(['agency', 'month'])
      .size()
      .unstack(fill_value=0)
)
pivot_full['TOTAL'] = pivot_full.sum(axis=1)
pivot_full.sort_values('TOTAL', ascending=False)

month,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,...,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,TOTAL
agency,,,,,,,,,,,,,,,,,,,,,
LAS,1,5,2,4,1,2,6,10,5,11,...,8,3,7,11,13,12,7,7,3,202
Blossom California Fertility,0,0,0,0,0,0,0,0,1,2,...,12,3,3,8,4,5,5,3,0,89
SurrogateFirst,2,1,2,1,3,1,3,1,1,3,...,3,5,4,3,2,3,6,1,1,82
Giving Tree Surrogacy,2,1,1,5,1,1,1,2,0,1,...,1,3,1,1,4,5,5,2,0,59
Indy,0,0,2,1,1,0,0,1,2,1,...,0,0,1,0,3,2,1,1,0,29
Twinkle Surrogacy,0,0,0,0,0,0,1,0,0,0,...,2,6,2,1,2,0,2,1,0,28
Flying Stork,0,1,1,0,1,0,0,1,1,0,...,0,1,0,0,0,1,1,2,0,23
C&T Fertility Consultant,0,0,0,0,0,0,0,0,0,1,...,1,2,3,5,0,1,1,1,0,22
Lily Baby,0,0,0,0,0,0,0,0,0,0,...,1,3,1,2,1,3,2,0,0,21


---
## Filter helpers — change the values below and re-run the cell

In [60]:
# ── Filter by month ───────────────────────────────────────────────────────────
MONTH = '2025-01'   # ← change to any YYYY-MM

filtered = df[df['month'] == MONTH][['date', 'agency', 'status', 'case_folder', 'agreement_file']]
print(f'{MONTH}: {len(filtered)} cases  '
      f'(drafting={( filtered["status"]=="drafting").sum()}, '
      f'review={(filtered["status"]=="review").sum()})')
filtered

2025-01: 42 cases  (drafting=29, review=13)


,date,agency,status,case_folder,agreement_file
279,2025-01-09,All Love,drafting,25-S-11 Mao & Villanueva (WA) - All Love Surrogacy,(Signed) Agreement for Legal Representation - Mao & Villanueva (WA) - All Love Surrogacy.pdf
280,2025-01-07,Babies Sprouts,review,25-S-13 Zhou and Zhu & Gamez - Babies Sprouts review,(Signed) Agreement for Legal Representation - Zhou and Zhu & Gamez - Babies Sprouts review.pdf
281,2025-01-22,Biggest Ask,review,24-S-319 Hernandez and Osuna - Biggest Ask review,Agreement for Legal Representation - Hernandez and Osuna - Biggest Ask - signed.pdf
282,2025-01-07,Blissful Bloom,drafting,24-S-309 Chen & Record (NJ) - Blissful Bloom,Agreement for Legal Representation - Chen & Record (NJ) - Blissful Bloom - signed.pdf
283,2025-01-05,Blossom California Fertility,review,24-S-318 Wang and Li & Gagnon (AZ) - Blossom California Fertility review,(Signed) Agreement for Legal Representation - Wang and Li & Gagnon (AZ) - Blossom California Fertility review.pdf
284,2025-01-15,Blossom California Fertility,review,25-S-18 Xu and Cao & Naranjo (WA) - Blossom California Fertility review,Agreement for Legal Representation- Xu and Cao & Naranjo (WA) - Blossom California Fertility review 01142025 - signe...
285,2025-01-21,Blossom California Fertility,review,25-S-25 Chen and Xu & Lina Ospina - Blossom California Fertility review,Agreement for Legal Representation - Chen and Xu & Gildardo - Blossom California Fertility review - signed.pdf
286,2025-01-22,Blossom California Fertility,review,25-S-27 Huang and Huang & Mercado-Flores - Blossom California Fertility review,Please Sign - Agreement for Legal Representation - Huang and Huang & Mercado-Flores - Blossom California review - si...
287,2025-01-22,Blossom California Fertility,drafting,25-S-33 Xu & Gloria (TX) - Blossom California Fertility,(signed) Agreement for Legal Representation - Xu & Gloria (TX) - Blossom California Fertility (2).pdf
288,2025-01-22,Carrying Dreams,drafting,25-S-20 Lugten and Bouter & Greytak (WA-AZ) - Carrying Dreams,Agreement for Legal Representation - Lugten and Bouter & Greytak (AZ-WA) - Carrying Dreams - signed.pdf


In [61]:
# ── Filter by agency (partial match, case-insensitive) ────────────────────────
AGENCY = 'LAS'   # ← change to any agency name or substring

filtered = df[df['agency'].str.contains(AGENCY, case=False, na=False)][['month', 'date', 'agency', 'status', 'case_folder', 'agreement_file']]
print(f'Matching "{AGENCY}": {len(filtered)} cases')
filtered

Matching "LAS": 202 cases


,month,date,agency,status,case_folder,agreement_file
7,2024-01,2024-01-06,LAS,drafting,24-S-02 Zhang & Garcia - LAS,Agreement for Legal Representation - Zhang & Garcia - LAS - signed.pdf
19,2024-02,2024-02-03,LAS,drafting,24-S-21 Yang and Wang & Ochoa - LAS,Agreement for Legal Representation - Yang and Wang & Ochoa - LAS - signed.pdf
20,2024-02,2024-02-05,LAS,drafting,24-S-19 Peng and Qian & Martinez - LAS,Agreement for Legal Representation - Peng and Qian & Martinez - LAS - signed.pdf
21,2024-02,2024-02-07,LAS,drafting,24-S-24 Ma and Zhang & Vega (FL) - LAS,Agreement for Legal Representation - Ma and Zhang & Vega (FL) - LAS - signed.pdf
22,2024-02,2024-02-14,LAS,drafting,24-S-27 Chiu & Garcia - LAS,(Signed) Agreement for Legal Representation - Chiu & Garcia - LAS.pdf
23,2024-02,2024-02-20,LAS,drafting,24-S-35 Feng and Zhao & Sandoval-Maynor - LAS,Agreement for Legal Representation - Feng and Zhao & Sandoval-Maynor - LAS - signed.pdf
36,2024-03,2024-03-12,LAS,drafting,24-S-50 Ghotb and Karabulut & Flores - LAS,Agreement for Legal Representation - Ghotb and Karabulut & Flores - LAS - signed.pdf
37,2024-03,2024-03-15,LAS,drafting,24-S-39 Wang and He & Lopez - LAS,Garcia-Lopez - GC Attorney Legal Clearance letter.pdf
61,2024-04,2024-04-02,LAS,drafting,24-S-68 Long & Gaspar Carranza - LAS,Agreement for Legal Representation - Long & Gaspar Carranza - LAS - Signed.pdf
62,2024-04,2024-04-18,LAS,drafting,24-S-49 Zhao & Garcia - LAS,Legal Clearance Letter - Yuhan Zhao & Adilene M. Garcia Rosales - LAS.pdf


In [62]:
# ── Filter by status ──────────────────────────────────────────────────────────
STATUS = 'review'   # ← 'review' or 'drafting'

filtered = df[df['status'] == STATUS][['month', 'date', 'agency', 'case_folder', 'agreement_file']]
print(f'Status={STATUS!r}: {len(filtered)} cases')
filtered

Status='review': 321 cases


,month,date,agency,case_folder,agreement_file
1,2024-01,2024-01-11,All Families Surrogacy,24-S-05 Santelli and Hagood & Severn - Review (WA) All Families Surrogacy,Agreement for Legal Representation - Santelli and Hagood & Severn review (WA) AFS - signed.pdf
2,2024-01,2024-01-22,California Surrogacy Partners,24-S-08 Latner and Feldstein & Freytag - Review CSP,Agreement for Legal Representation - Latner and Feldstein & Freytag - Review - CSP - IM Signed.pdf
3,2024-01,2024-01-08,Giving Tree Surrogacy,24-S-03 Ding & Martinez - Giving Tree Surrogacy (Review),Agreement for Legal Representation- Martinez GTS review - signed.pdf
4,2024-01,2024-01-18,Giving Tree Surrogacy,24-S-07 Watters and Mastrodonato & Carli - Review - GTS,Agreement for Legal Representation - Watters and Mastrodonato & Carli - Review - GTS - signed.pdf
8,2024-01,2024-01-22,Pitter Patter,24-S-10 Werner review Pitter Patter,Agreement for Legal Representation - Jovanoic & Werner - Pitter Patter review - Signed.pdf
9,2024-01,2024-01-31,Pitter Patter,24-S-18 Corradi & DeBiase - Review Pitter Patter,Agreement for Legal Representation - Sierra DeBiase - Review - Pitter Patter - signed.pdf
13,2024-02,2024-02-27,California Surrogacy Partners,24-S-26 Kohler and Richards & Nelson UT review CSP,IP Legal Clearance Letter.pdf
14,2024-02,2024-02-29,Dreamweaver,24-S-41 Zhuang and Abrams & Butteweg -Dreamweaver review (AZ),GC -Agreement for Legal Representation - Butteweg (AZ) - signed.pdf
17,2024-02,2024-02-21,Giving Tree Surrogacy,24-S-31 Chang and Hedges review GTS,Agreement for Legal Representation - Chang & Hedges review (AZ) GTS signed.pdf
24,2024-02,2024-02-02,Surrogacy.is,24-S-17 Bell & Smith review Surrogacy.is,Agreement for Legal Representation - Smith and Hyler review - Surrogacy.is - signed.pdf


In [63]:
# ── Search case folder name ───────────────────────────────────────────────────
SEARCH = '25-S-14'   # ← any substring of the folder name

df[df['case_folder'].str.contains(SEARCH, case=False, na=False)]

,month,date,agency,status,case_folder,agreement_file,source,path_len
305,2025-01,2025-01-09,LAS,drafting,25-S-14 Haoxue Zhu and Yifeng Cao & Jennifer Marisol Rivera Quntero - LAS,(Signed) Agreement for Legal Representation - Haoxue Zhu and Yifeng Cao & Jennifer Rivera Quntero - LAS.pdf,signed agreement,274
389,2025-04,2025-04-23,Blissful Bloom,drafting,25-S-141 Yu and Liu & Lee - Blissful Bloom,Agreement for Legal Representation - Yu and Liu & Lee - Blissful Bloom - signed.pdf,signed agreement,219
395,2025-04,2025-04-26,Blossom California Fertility,review,25-S-144 Luo and Lyu & Cupp (TX) - Blossom California Fertility review,(Signed) Agreement for Legal Representation - Luo and Lyu & Cupp (TX) - Blossom California Fertility review.pdf,signed agreement,275
403,2025-04,2025-04-23,Giving Tree Surrogacy,drafting,25-S-143 Lu and Huang & Lamin (OH) - GTS,请签署Agreement for Legal Representation - Lu and Huang & Lamin (OH) - GTS - signed.pdf,signed agreement,219
410,2025-04,2025-04-28,LAS,drafting,25-S-146 Pei-Yu Chien and Chun-Yi Wu & Cecilia Perez - LAS,Agreement for Legal Representation - Pei-Yu Chien and Chun-Yi Wu & Cecilia Perez - LAS - Signed.pdf,signed agreement,251
411,2025-04,2025-04-30,LAS,drafting,25-S-147 Xiuhong Hu and Chao-Yang Chan & Cassandra Moreno - LAS,请签署Agreement for Legal Representation - Xiuhong Hu and Chao-Yang Chan & Cassandra Moreno - LAS - signed.pdf,signed agreement,264
412,2025-04,2025-04-24,Lily Baby,drafting,25-S-140 Guo and Zhu & Perez (IL) - Lily Baby Surrogacy,(Signed) Agreement for Legal Representation - Guo and Zhu & Perez (IL) - Lily Baby Surrogacy_2.pdf,signed agreement,247
422,2025-04,2025-04-30,SurrogateFirst,drafting,25-S-149 Hammond III and Weyker (NY) - SurrogateFirst,Agreement for Legal Representation - Hammond III and Weyker (NY) - SurrogateFirst - signed.pdf,signed agreement,241
430,2025-05,2025-05-06,Dream Surrogacy,drafting,25-S-145 Moore & Aaron - Dream Surrogacy,Agreement for Legal Representation - Moore & Aaron - Dream Surrogacy - signed.pdf,signed agreement,215
442,2025-05,2025-05-02,LAS,review,25-S-148 Jia Li & Dailenea Heffron - LAS review,Agreement for Legal Representation - Jia Li & Dailenea Heffron - LAS review - signed.pdf,signed agreement,229


In [64]:
# ── All cases sorted by case number (24-S-01 first, most recent at bottom) ────

def _case_sort_key(folder):
    m = re.match(r'^(\d{2})-[A-Za-z]+-?(\d+)', folder)
    return (int(m.group(1)), int(m.group(2))) if m else (99, 99999)

sorted_df = df.copy()
sorted_df['_key'] = sorted_df['case_folder'].apply(_case_sort_key)
sorted_df = (sorted_df
             .sort_values('_key')
             .drop(columns='_key')
             .reset_index(drop=True))
sorted_df.index = range(1, len(sorted_df) + 1)
sorted_df.index.name = '#'
print(f'Total: {len(sorted_df)} cases')
sorted_df[['month', 'date', 'agency', 'status', 'case_folder', 'agreement_file']]

Total: 1090 cases


,month,date,agency,status,case_folder,agreement_file
#,,,,,,
1,2024-01,2024-01-08,Kindred Surrogacy,drafting,24-S-01 Wu and Yang & Wilson (CA) - Kindred Surrogacy,Agreement for Legal Representation - Wu and Yang & Wilson (CA) Kindred Surrogacy - signed.pdf
2,2024-01,2024-01-06,LAS,drafting,24-S-02 Zhang & Garcia - LAS,Agreement for Legal Representation - Zhang & Garcia - LAS - signed.pdf
3,2024-01,2024-01-08,Giving Tree Surrogacy,review,24-S-03 Ding & Martinez - Giving Tree Surrogacy (Review),Agreement for Legal Representation- Martinez GTS review - signed.pdf
4,2024-01,2024-01-12,SurrogateFirst,drafting,24-S-04 Wang & Moritz (WA) SurrogateFirst,Agreement for Legal Representation - Wang & Mortiz (WA) SurrogateFirst - signed.pdf
5,2024-01,2024-01-11,All Families Surrogacy,review,24-S-05 Santelli and Hagood & Severn - Review (WA) All Families Surrogacy,Agreement for Legal Representation - Santelli and Hagood & Severn review (WA) AFS - signed.pdf
...,...,...,...,...,...,...
1086,2026-06,2026-06-09,EDSI,drafting,26-S-281 Gottpati and Shiak & Rhoades (AZ) - EDSI,Agreement for Legal Representation - Gottpati and Shiak & Rhoades (AZ) - EDSI - signed.pdf
1087,2026-06,2026-06-10,The Fertility Solutions,drafting,26-S-282 Ma and Fang & Tonkovich - The Fertility Solutions LLC,(Signed) Agreement for Legal Representation - Ma and Fang & Tonkovich - The Fertility Solutions LLC.pdf
1088,2026-06,2026-06-10,One Village,drafting,26-S-283 Hung & Ward (KY) - One Village Surrogacy,请签署Agreement for Legal Representation - Hung & Ward (KY) - One Village Surrogacy - signed.pdf


In [65]:
# ── Case folders still missing (no signed agreement and no clearance letter) ───

def _case_sort_key(folder):
    m = re.match(r'^(\d{2})-[A-Za-z]+-?(\d+)', folder)
    return (int(m.group(1)), int(m.group(2))) if m else (99, 99999)

def _missing_reason(folder_name, files):
    fn = folder_name.lower()
    if 'terminat' in fn:
        return 'case terminated'
    if 'not to represent' in fn or 'decide not' in fn or 'declined' in fn:
        return 'decided not to represent'
    if not files:
        return 'empty folder'
    img_exts = {'.pdf', '.jpg', '.jpeg', '.png', '.gif', '.tiff', '.bmp'}
    has_agreement_pdf = any(
        any(f.lower().endswith(e) for e in img_exts)
        and 'agreement' in f.lower() and 'legal' in f.lower() and 'representation' in f.lower()
        for f in files
    )
    if has_agreement_pdf:
        return 'agreement exists but not marked as signed'
    if any('agreement' in f.lower() for f in files):
        return 'agreement file found but wrong format/name'
    return 'no agreement file in folder'

missing = []

for folder_name in sorted(os.listdir(BASE_DIR)):
    if folder_name.startswith('.') or folder_name.startswith('~'):
        continue
    folder_path = os.path.join(BASE_DIR, folder_name)
    if not os.path.isdir(folder_path):
        continue
    if not (folder_name.startswith('24-') or folder_name.startswith('25-') or folder_name.startswith('26-')):
        continue
    parsed = parse_folder_name(folder_name)
    if not parsed:
        continue
    raw_agency, _ = parsed
    if not normalize_agency_name(raw_agency):
        continue
    if find_agreement_file(folder_path) or find_clearance_letter(folder_path):
        continue  # already counted in df

    case_no = folder_name.split()[0]
    try:
        files = sorted(e.name for e in os.scandir(folder_path) if e.is_file())
    except:
        files = []

    missing.append({
        'case_no':     case_no,
        'case_folder': folder_name,
        'reason':      _missing_reason(folder_name, files),
        'files':       '\n'.join(files) if files else '(empty folder)',
    })

miss_df = pd.DataFrame(missing)
if len(miss_df):
    miss_df['_sort'] = miss_df['case_no'].apply(lambda c: _case_sort_key(c + ' '))
    miss_df = miss_df.sort_values('_sort').drop(columns='_sort').reset_index(drop=True)
    miss_df.index = range(1, len(miss_df) + 1)
    miss_df.index.name = '#'

print(f'Still missing (no agreement and no clearance letter): {len(miss_df)}')
pd.set_option('display.max_rows', 2000)
pd.set_option('display.max_colwidth', 300)
miss_df

Still missing (no agreement and no clearance letter): 72


,case_no,case_folder,reason,files
#,,,,
1,24-S-11,24-S-11 Uribe & Bowman - LAS,empty folder,(empty folder)
2,24-S-11,24-S-11 Uribe & Bowman - LAS - TERMINATE4D,case terminated,Agreement for Legal Representation - Uribe & Bowman - LAS.docx\nAgreement for Legal Representation - Uribe & Bowman - LAS.pdf\nFIRST DRAFT Gestational Surrogacy Agreement - Uribe & Bowman - LAS 01282024.docx\nFIRST DRAFT Gestational Surrogacy Agreement - Uribe & Bowman - LAS 01282024.pdf\nIntern...
3,24-S-23,24-S-23 (ON HOLD) Independent Surrogacy - Yang & Shevshenko,agreement exists but not marked as signed,(Internal Drafting) Gestational Surrogacy Agreement - Yang & Shevshenko - independent.docx\nAgreement for Legal Representation - Yang & Shevshenko.docx\nAgreement for Legal Representation - Yang & Shevshenko.pdf\nCompleted Match Sheet - Yang - Independant Surrogacy Match.pdf
4,24-S-29,24-S-29 Wang and Chen & Willhite BBB (MO) nmc,empty folder,(empty folder)
5,24-S-32,24-S-32 Kazmi & Zachariadis and Gutierrez PBO review Surroconnections,agreement exists but not marked as signed,Agreement for Legal Representation - Kazmi & Zachariadis and Gutierrez PBO review Surroconnections.docx\nAgreement for Legal Representation - Kazmi & Zachariadis and Gutierrez PBO review Surroconnections.pdf\nAgreement for Legal Representation.doc\nAhmed.png\nCertified Judgment - Kazmi and Zacha...
6,24-S-46,24-S-46 Dollinger & Soto - Reproductive Options review,agreement exists but not marked as signed,Agreement for Legal Representation - Dollinger & Soto .docx\nAgreement for Legal Representation - Dollinger & Soto .pdf\nGC Maria match sheet.pdf
7,24-S-51,24-S-51 Angwang and Velez review - New Gen (AZ),agreement exists but not marked as signed,Agreement for Legal Representation - Angwang and Velez review - New Gen (AZ).docx\nAgreement for Legal Representation - Angwang and Velez review - New Gen (AZ).pdf\nGestational Surrogacy Agreement - Angwang and Velez review - New Gen (AZ).pdf\nLuozhu & Li - Raquel & Austin.GS.d1.docx\nLuozhu & L...
8,24-S-54,24-S-54 Li and Cheng & Sanchez - LAS - NEED A REMATCH,no agreement file in folder,"Flor S. Benefit package (signed).pdf\nFlor Sanchez, Match sheet.pdf\nMatch Info.pdf\nunnamed.jpg"
9,24-S-76,24-S-76 Sampat & Dorman - Pitter Patter (FL),agreement exists but not marked as signed,Agreement for Legal Representation -Sampat.docx\nAgreement for Legal Representation -Sampat.pdf\nBryanne_Benefit_Packet-bryannedorman_gmail.com.pdf\nCompare GSA Sampat.pdf\nCompare Internal Draft -Gestational Surrogacy Agreement - Sampat & Dorman Pitter Patter.docx\nDRAFT Gestational Surrogacy A...


In [66]:
# ── Export to cases stats folder ───────────────────────────────────────────────

OUT_DIR = r"C:\Users\zhous\OneDrive - Tsong Law Group\Ralph Tsong's files - Marketing\cases stats"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Rebuild summaries ──────────────────────────────────────────────────────────
_pivot_month = (
    df.groupby(['month', 'status'])
      .size().unstack(fill_value=0)
      .reindex(columns=['drafting', 'review'], fill_value=0)
)
_pivot_month['total'] = _pivot_month.sum(axis=1)
_pivot_month.loc['TOTAL'] = _pivot_month.sum()

_pivot_agency = (
    df.groupby(['agency', 'status'])
      .size().unstack(fill_value=0)
      .reindex(columns=['drafting', 'review'], fill_value=0)
)
_pivot_agency['total'] = _pivot_agency.sum(axis=1)
_pivot_agency = _pivot_agency.sort_values('total', ascending=False)

_pivot_agency_month = (
    df.groupby(['agency', 'month'])
      .size().unstack(fill_value=0)
)
_pivot_agency_month['TOTAL'] = _pivot_agency_month.sum(axis=1)
_pivot_agency_month = _pivot_agency_month.sort_values('TOTAL', ascending=False)

_all_cases = df[['month', 'date', 'agency', 'status', 'source', 'case_folder', 'agreement_file']].copy()
_all_cases.index = range(1, len(_all_cases) + 1)
_all_cases.index.name = '#'

# ── HTML style ─────────────────────────────────────────────────────────────────
_CSS = """
<style>
  body { font-family: Arial, sans-serif; font-size: 13px; margin: 20px; }
  h2   { color: #2c3e50; }
  table { border-collapse: collapse; width: 100%; }
  th   { background: #2c3e50; color: white; padding: 6px 10px; text-align: left; }
  td   { padding: 5px 10px; border-bottom: 1px solid #ddd; white-space: pre-wrap; }
  tr:nth-child(even) { background: #f5f5f5; }
</style>
"""

def _to_html(title, frame):
    return f"<html><head><meta charset='utf-8'>{_CSS}</head><body><h2>{title}</h2>{frame.to_html()}</body></html>"

# ── Write files ────────────────────────────────────────────────────────────────
exports = {
    'all_cases':             (_all_cases,           'All Cases (1 row per case)'),
    'monthly_summary':       (_pivot_month,          'Monthly Summary'),
    'agency_summary':        (_pivot_agency,         'Agency Summary'),
    'agency_x_month':        (_pivot_agency_month,   'Agency × Month Breakdown'),
    'missing_cases':         (miss_df,               'Missing Cases (no agreement / clearance letter)'),
}

for fname, (frame, title) in exports.items():
    csv_path  = os.path.join(OUT_DIR, f'{fname}.csv')
    html_path = os.path.join(OUT_DIR, f'{fname}.html')
    frame.to_csv(csv_path, encoding='utf-8-sig')
    with open(html_path, 'w', encoding='utf-8') as f:
        f.write(_to_html(title, frame))
    print(f'✓  {fname}.csv  +  {fname}.html')

print(f'\nSaved to: {OUT_DIR}')

✓  all_cases.csv  +  all_cases.html
✓  monthly_summary.csv  +  monthly_summary.html
✓  agency_summary.csv  +  agency_summary.html
✓  agency_x_month.csv  +  agency_x_month.html
✓  missing_cases.csv  +  missing_cases.html

Saved to: C:\Users\zhous\OneDrive - Tsong Law Group\Ralph Tsong's files - Marketing\cases stats


In [67]:
# ── Interactive charts — setup ─────────────────────────────────────────────────
%pip install -q plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

OUT_DIR = r"C:\Users\zhous\OneDrive - Tsong Law Group\Ralph Tsong's files - Marketing\cases stats"
_COLORS = {'drafting': '#2c7bb6', 'review': '#d7191c'}

# ── 1. Monthly case volume ─────────────────────────────────────────────────────
_m = (df.groupby(['month', 'status']).size().unstack(fill_value=0)
        .reindex(columns=['drafting', 'review'], fill_value=0))
_m['total'] = _m.sum(axis=1)

fig1 = go.Figure()
fig1.add_bar(x=_m.index, y=_m['drafting'], name='Drafting',
             marker_color=_COLORS['drafting'],
             text=_m['drafting'], textposition='inside')
fig1.add_bar(x=_m.index, y=_m['review'], name='Review',
             marker_color=_COLORS['review'],
             text=_m['review'], textposition='inside')
fig1.add_scatter(x=_m.index, y=_m['total'], name='Total',
                 mode='lines+markers', line=dict(color='#333', width=2),
                 marker=dict(size=6))
fig1.update_layout(
    title='Monthly Case Volume',
    barmode='stack', xaxis_title='Month', yaxis_title='Cases',
    hovermode='x unified', template='plotly_white',
    legend=dict(orientation='h', y=1.08)
)
fig1.show()

Note: you may need to restart the kernel to use updated packages.


In [68]:
# ── 2. Top 30 agencies — pie chart ────────────────────────────────────────────
_a = (df.groupby(['agency', 'status']).size().unstack(fill_value=0)
        .reindex(columns=['drafting', 'review'], fill_value=0))
_a['total'] = _a.sum(axis=1)
_a = _a.sort_values('total', ascending=False).head(30)

fig2 = go.Figure(go.Pie(
    labels=_a.index,
    values=_a['total'],
    textinfo='label+percent+value',
    hovertemplate='%{label}<br>Cases: %{value}<br>%{percent}<extra></extra>',
))
fig2.update_layout(
    title='Top 30 Agencies by Case Count',
    template='plotly_white',
    height=800,
)
fig2.show()

In [69]:
# ── 3. Agency × Month heatmap (top 30 agencies) ───────────────────────────────
_h = (df.groupby(['agency', 'month']).size().unstack(fill_value=0))
_h['_total'] = _h.sum(axis=1)
_h = _h.sort_values('_total', ascending=False).head(30).drop(columns='_total')

fig3 = go.Figure(go.Heatmap(
    z=_h.values,
    x=_h.columns.tolist(),
    y=_h.index.tolist(),
    colorscale='Blues',
    text=_h.values,
    texttemplate='%{text}',
    hovertemplate='Agency: %{y}<br>Month: %{x}<br>Cases: %{z}<extra></extra>',
))
fig3.update_layout(
    title='Agency × Month Heatmap (Top 30 Agencies)',
    xaxis_title='Month', yaxis_title='',
    template='plotly_white', height=800,
    xaxis=dict(tickangle=-45)
)
fig3.show()

In [70]:
# ── 4. Cumulative cases over time ─────────────────────────────────────────────
_cum = _m[['drafting', 'review', 'total']].cumsum()

fig4 = go.Figure()
fig4.add_scatter(x=_cum.index, y=_cum['total'], name='Total',
                 fill='tozeroy', line=dict(color='#555', width=2))
fig4.add_scatter(x=_cum.index, y=_cum['drafting'], name='Drafting',
                 line=dict(color=_COLORS['drafting'], width=2, dash='dot'))
fig4.add_scatter(x=_cum.index, y=_cum['review'], name='Review',
                 line=dict(color=_COLORS['review'], width=2, dash='dot'))
fig4.update_layout(
    title='Cumulative Cases Over Time',
    xaxis_title='Month', yaxis_title='Cumulative Cases',
    template='plotly_white', hovermode='x unified',
    legend=dict(orientation='h', y=1.08)
)
fig4.show()

In [71]:
# ── 5. Source breakdown — donut ───────────────────────────────────────────────
_src = df['source'].value_counts()
fig5 = go.Figure(go.Pie(
    labels=_src.index, values=_src.values,
    hole=0.45,
    marker_colors=['#2c7bb6', '#abd9e9'],
    textinfo='label+percent+value',
))
fig5.update_layout(
    title='Cases by Source (Signed Agreement vs Clearance Letter)',
    template='plotly_white'
)
fig5.show()

In [72]:
# ── 6. Agency time series — drafting vs review by month (interactive dropdown) ─

# Monthly counts per agency + status
_ts = (
    df.groupby(['agency', 'month', 'status'])
      .size()
      .unstack(fill_value=0)
      .reindex(columns=['drafting', 'review'], fill_value=0)
      .reset_index()
)

# Agencies sorted by total cases descending (most active first in dropdown)
_agency_order = df.groupby('agency').size().sort_values(ascending=False).index.tolist()

fig6 = go.Figure()

# Two traces per agency (drafting + review); only the first agency is visible
for i, agency in enumerate(_agency_order):
    ag = _ts[_ts['agency'] == agency].sort_values('month')
    vis = (i == 0)
    fig6.add_scatter(
        x=ag['month'], y=ag['drafting'],
        name='Drafting', mode='lines+markers',
        line=dict(color=_COLORS['drafting'], width=2),
        marker=dict(size=7), visible=vis,
        hovertemplate='%{x}<br>Drafting: %{y}<extra></extra>',
    )
    fig6.add_scatter(
        x=ag['month'], y=ag['review'],
        name='Review', mode='lines+markers',
        line=dict(color=_COLORS['review'], width=2),
        marker=dict(size=7), visible=vis,
        hovertemplate='%{x}<br>Review: %{y}<extra></extra>',
    )

# Dropdown: each button shows only 2 traces for the selected agency
n = len(_agency_order)
_buttons = []
for i, agency in enumerate(_agency_order):
    vis_list = [False] * (2 * n)
    vis_list[2 * i]     = True  # drafting
    vis_list[2 * i + 1] = True  # review
    _buttons.append(dict(
        label=agency,
        method='update',
        args=[
            {'visible': vis_list},
            {'title': f'Monthly Cases Over Time — {agency}'},
        ],
    ))

fig6.update_layout(
    title=f'Monthly Cases Over Time — {_agency_order[0]}',
    xaxis_title='Month',
    yaxis_title='Cases',
    template='plotly_white',
    height=520,
    hovermode='x unified',
    legend=dict(orientation='h', y=1.14),
    updatemenus=[dict(
        buttons=_buttons,
        direction='down',
        showactive=True,
        x=0.0,
        xanchor='left',
        y=1.25,
        yanchor='top',
        bgcolor='white',
        bordercolor='#ccc',
    )],
)
fig6.show()

In [73]:
# ── Save all charts to cases stats folder ────────────────────────────────────
charts = {
    'chart_monthly_volume':    fig1,
    'chart_top_agencies':      fig2,
    'chart_agency_heatmap':    fig3,
    'chart_cumulative':        fig4,
    'chart_source_breakdown':  fig5,
    'chart_agency_timeseries': fig6,
}

for fname, fig in charts.items():
    fig.write_html(f'{OUT_DIR}\\{fname}.html', include_plotlyjs='cdn')
    print(f'✓  {fname}.html')

print(f'\nSaved to: {OUT_DIR}')

✓  chart_monthly_volume.html
✓  chart_top_agencies.html
✓  chart_agency_heatmap.html
✓  chart_cumulative.html
✓  chart_source_breakdown.html
✓  chart_agency_timeseries.html

Saved to: C:\Users\zhous\OneDrive - Tsong Law Group\Ralph Tsong's files - Marketing\cases stats
